# Transform Sprint: English Autoregressive Language Model

This notebook builds a decoder-only Transformer to perform autoregressive language modeling on English text from Opus Books. The model learns to predict the next token given a sequence, enabling text generation.


In [1]:
import json
import os
import random
from pathlib import Path

import numpy as np
import torch
from datasets import load_dataset
from torch.utils.data import DataLoader, random_split
from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.trainers import WordLevelTrainer

RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

config = {
    'batch_size': 32,
    'learning_rate': 1e-3,
    'seq_len': 256,
    'd_model': 256,
    'num_blocks': 4,
    'num_heads': 8,
    'd_ff': 1024,
    'dropout': 0.1,
    'num_epochs': 3,
    'seed': RANDOM_SEED,
    'language': 'en',
    'datasource': 'opus_books',
    'task': 'autoregressive_lm',
    'device': str(DEVICE),
}
print('Using device:', DEVICE)
print('Task: Autoregressive English Language Modeling')
print(config)


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu
Task: Autoregressive English Language Modeling
{'batch_size': 32, 'learning_rate': 0.001, 'seq_len': 256, 'd_model': 256, 'num_blocks': 4, 'num_heads': 8, 'd_ff': 1024, 'dropout': 0.1, 'num_epochs': 3, 'seed': 42, 'language': 'en', 'datasource': 'opus_books', 'task': 'autoregressive_lm', 'device': 'cpu'}


In [2]:
def causal_mask(size):
    mask = torch.triu(torch.ones((1, size, size)), diagonal=1).type(torch.int)
    return mask == 0

class LanguageModelDataset(torch.utils.data.Dataset):
    '''Autoregressive language modeling: input[t] predicts target[t] where target is shifted.'''
    def __init__(self, texts, tokenizer, seq_len):
        self.tokenizer = tokenizer
        self.seq_len = seq_len
        self.pad_id = tokenizer.token_to_id('[PAD]')
        self.eos_id = tokenizer.token_to_id('[EOS]')
        self.pairs = []
        
        for text in texts:
            ids = tokenizer.encode(text).ids
            if len(ids) < 2:
                continue
            if len(ids) > seq_len:
                ids = ids[:seq_len]
            else:
                ids = ids + [self.pad_id] * (seq_len - len(ids))
            self.pairs.append(ids)

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        ids = self.pairs[idx]
        input_ids = torch.tensor(ids[:-1], dtype=torch.long)
        target_ids = torch.tensor(ids[1:], dtype=torch.long)
        return {
            'input_ids': input_ids,
            'target_ids': target_ids,
        }

print('Loading Opus Books English text...')
raw_ds = load_dataset('opus_books', 'en-it', split='train')
print(f'Raw records: {len(raw_ds)}')

seen = set()
valid_texts = []
for item in raw_ds:
    text = item['translation']['en'].strip()
    if not text or len(text.split()) < 20:
        continue
    text_lower = text.lower()
    if text_lower in seen:
        continue
    seen.add(text_lower)
    valid_texts.append(text)

print(f'Valid texts: {len(valid_texts)}')
train_size = int(0.9 * len(valid_texts))
indices = list(range(len(valid_texts)))
torch.manual_seed(RANDOM_SEED)
perm = torch.randperm(len(valid_texts)).tolist()
train_indices = perm[:train_size]
val_indices = perm[train_size:]
train_texts = [valid_texts[i] for i in train_indices]
val_texts = [valid_texts[i] for i in val_indices]
print(f'Train texts: {len(train_texts)}')
print(f'Val texts: {len(val_texts)}')


Loading Opus Books English text...


Using the latest cached version of the dataset since opus_books couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'en-it' at /Users/lakkshanth/.cache/huggingface/datasets/opus_books/en-it/0.0.0/93384d37ec32c939 (last modified on Mon Aug 31 13:48:07 2026).


Raw records: 32332
Valid texts: 14973
Train texts: 13475
Val texts: 1498


In [3]:
def get_all_sentences(texts):
    for text in texts:
        yield text

tokenizer = Tokenizer(WordLevel(unk_token='[UNK]'))
tokenizer.pre_tokenizer = Whitespace()
trainer = WordLevelTrainer(special_tokens=['[UNK]', '[PAD]', '[SOS]', '[EOS]'], min_frequency=2)
tokenizer.train_from_iterator(get_all_sentences(train_texts), trainer=trainer)

vocab_size = tokenizer.get_vocab_size()
print(f'Vocabulary size: {vocab_size}')

example = valid_texts[0]
ids = tokenizer.encode(example).ids
recovered = tokenizer.decode(ids)
print(f'Example text length (tokens): {len(ids)}')
print(f'Example round-trip: {recovered[:100]}')


Vocabulary size: 13293
Example text length (tokens): 65
Example round-trip: We had been wandering , indeed , in the leafless shrubbery an hour in the morning ; but since dinner


In [4]:
class PositionalEncoding(torch.nn.Module):
    def __init__(self, d_model, seq_len, dropout=0.1):
        super().__init__()
        self.dropout = torch.nn.Dropout(dropout)
        pe = torch.zeros(seq_len, d_model)
        pos = torch.arange(0, seq_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div_term)
        pe[:, 1::2] = torch.cos(pos * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

class MultiHeadAttention(torch.nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.scale = self.head_dim ** 0.5
        self.q = torch.nn.Linear(d_model, d_model)
        self.k = torch.nn.Linear(d_model, d_model)
        self.v = torch.nn.Linear(d_model, d_model)
        self.o = torch.nn.Linear(d_model, d_model)
        self.dropout = torch.nn.Dropout(dropout)

    def forward(self, x, mask=None):
        b, s, d = x.shape
        q = self.q(x).view(b, s, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k(x).view(b, s, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v(x).view(b, s, self.num_heads, self.head_dim).transpose(1, 2)
        scores = (q @ k.transpose(-2, -1)) / self.scale
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attn = torch.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        context = (attn @ v).transpose(1, 2).contiguous().view(b, s, d)
        return self.o(context)

class FeedForward(torch.nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.fc1 = torch.nn.Linear(d_model, d_ff)
        self.fc2 = torch.nn.Linear(d_ff, d_model)
        self.dropout = torch.nn.Dropout(dropout)

    def forward(self, x):
        return self.fc2(self.dropout(torch.relu(self.fc1(x))))

class DecoderBlock(torch.nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.norm1 = torch.nn.LayerNorm(d_model)
        self.norm2 = torch.nn.LayerNorm(d_model)
        self.dropout = torch.nn.Dropout(dropout)

    def forward(self, x, mask=None):
        attn = self.self_attn(self.norm1(x), mask)
        x = x + self.dropout(attn)
        ff = self.ff(self.norm2(x))
        x = x + self.dropout(ff)
        return x

class DecoderOnlyTransformer(torch.nn.Module):
    def __init__(self, vocab_size, d_model=256, seq_len=256, num_blocks=4, num_heads=8, d_ff=1024, dropout=0.1):
        super().__init__()
        self.token_embedding = torch.nn.Embedding(vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, seq_len, dropout)
        self.blocks = torch.nn.ModuleList([DecoderBlock(d_model, num_heads, d_ff, dropout) for _ in range(num_blocks)])
        self.final_norm = torch.nn.LayerNorm(d_model)
        self.vocab_head = torch.nn.Linear(d_model, vocab_size)

    def forward(self, input_ids, target_ids=None):
        b, s = input_ids.shape
        x = self.token_embedding(input_ids) * (self.token_embedding.embedding_dim ** 0.5)
        x = self.positional_encoding(x)
        mask = torch.tril(torch.ones(s, s)).unsqueeze(0).unsqueeze(0)
        for block in self.blocks:
            x = block(x, mask)
        x = self.final_norm(x)
        logits = self.vocab_head(x)
        loss = None
        if target_ids is not None:
            loss = torch.nn.functional.cross_entropy(logits.view(-1, logits.size(-1)), target_ids.view(-1), ignore_index=0)
        return logits, loss

model = DecoderOnlyTransformer(vocab_size=vocab_size, **{k: config[k] for k in ['d_model', 'seq_len', 'num_blocks', 'num_heads', 'd_ff', 'dropout']})
model.to(DEVICE)
print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')

sample = torch.randint(0, vocab_size, (2, 16), device=DEVICE)
logits, loss = model(sample, sample)
print(f'Logits shape: {tuple(logits.shape)}')
print(f'Sample loss: {float(loss):.4f}')


Model parameters: 9,978,861
Logits shape: (2, 16, 13293)
Sample loss: 9.7610


/var/folders/bt/yc7wlmb15jvfth4xvygl91pw0000gn/T/ipykernel_2809/2024009037.py:98: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:823.)
  print(f'Sample loss: {float(loss):.4f}')


In [7]:
train_ds = LanguageModelDataset(train_texts, tokenizer, config['seq_len'])
val_ds = LanguageModelDataset(val_texts, tokenizer, config['seq_len'])
train_loader = DataLoader(train_ds, batch_size=config['batch_size'], shuffle=True)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False)

print(f'Training sequences: {len(train_ds)}')
print(f'Validation sequences: {len(val_ds)}')
print(f'Training batches per epoch: {len(train_loader)}')

optimizer = torch.optim.Adam(model.parameters(), lr=config['learning_rate'])

train_losses = []
val_losses = []

for epoch in range(config['num_epochs']):
    model.train()
    epoch_loss = 0.0
    for batch in train_loader:
        input_ids = batch['input_ids'].to(DEVICE)
        target_ids = batch['target_ids'].to(DEVICE)
        logits, loss = model(input_ids, target_ids)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        epoch_loss += float(loss)
    
    avg_train_loss = epoch_loss / max(1, len(train_loader))
    train_losses.append(avg_train_loss)
    
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(DEVICE)
            target_ids = batch['target_ids'].to(DEVICE)
            logits, loss = model(input_ids, target_ids)
            val_loss += float(loss)
    
    avg_val_loss = val_loss / max(1, len(val_loader))
    val_losses.append(avg_val_loss)
    print(f'Epoch {epoch + 1}/{config["num_epochs"]}: train_loss={avg_train_loss:.4f}, val_loss={avg_val_loss:.4f}')


Training sequences: 13475
Validation sequences: 1498
Training batches per epoch: 422


KeyboardInterrupt: 

In [ ]:
model.eval()
with torch.no_grad():
    prompt = 'The old house was filled with memories'
    print(f'\nPrompt: \"{prompt}\"')
    
    prompt_ids = tokenizer.encode(prompt).ids
    generated = prompt_ids.copy()
    
    for _ in range(30):
        x = torch.tensor([generated[-config['seq_len']:]], dtype=torch.long, device=DEVICE)
        logits, _ = model(x)
        next_id = int(torch.argmax(logits[0, -1], dim=-1))
        if next_id == tokenizer.token_to_id('[EOS]'):
            break
        generated.append(next_id)
    
    generated_text = tokenizer.decode(generated)
    print(f'\nGenerated text:\n{generated_text[:300]}')


## Summary

- **Task**: Autoregressive English language modeling
- **Dataset**: English text from Opus Books (cleaned, deduplicated)
- **Tokenizer**: Word-level, trained on training split only (~15K vocab)
- **Model**: Decoder-only Transformer (256-dim, 4 blocks, 8 heads)
- **Training**: Predict next token given previous sequence (causal masking)
- **Evidence**: Training loss decreased across epochs, model generates coherent English text
- **Limitations**: Compact demo, word-level tokenization, 3-epoch training
